In [1]:
import sys
import os
from pathlib import Path
project_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "config.py").exists()
)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


In [2]:

import data_pipeline.data_scraper as ds
import config as cdr
import data_cleaner as dc
import database as db

Database folder path: D:\Masai\CapstoneProject\Zepto-Data-AI-Platform\data_pipeline\database


In [3]:
raw_books_data, selected_categories, raw_books_shape = ds.scrape_books()
print(f"Raw books data shape: {raw_books_data.shape}")
print(f"Selected categories: {selected_categories}")

Raw books data shape: (142, 5)
Selected categories: {'Mystery': {'url': 'https://books.toscrape.com/catalogue/category/books/mystery_3/index.html', 'book_count': 32}, 'Sequential Art': {'url': 'https://books.toscrape.com/catalogue/category/books/sequential-art_5/index.html', 'book_count': 75}, 'Romance': {'url': 'https://books.toscrape.com/catalogue/category/books/romance_8/index.html', 'book_count': 35}}


In [4]:
clean_data = dc.clean_books_data(raw_books_data)

print(f"Cleaned books data shape: {clean_data.shape}")
print(clean_data.head())    

Cleaned books data shape: (142, 6)
  category                                            title  rating  \
0  Mystery                                    Sharp Objects       4   
1  Mystery                             In a Dark, Dark Wood       1   
2  Mystery                              The Past Never Ends       4   
3  Mystery                                 A Murder in Time       1   
4  Mystery  The Murder of Roger Ackroyd (Hercule Poirot #4)       4   

   price_gbp  availability  price_inr  
0      47.82          True   5045.010  
1      19.63          True   2070.965  
2      56.50          True   5960.750  
3      16.64          True   1755.520  
4      44.10          True   4652.550  


In [5]:
clean_data.to_csv("D:\\Masai\\CapstoneProject\\Zepto-Data-AI-Platform\\data_pipeline\\output_data\\cleaned_data.csv", index=False)   


In [6]:
category_dataframe, availability_dataframe, books_dataframe=db.normalise_and_insert_data(dataframe=clean_data)

Database connected!
Required tables created successfully!
Database connection closed!
Database connected!
3 records inserted into 'category_master'.
1 records inserted into 'availability_master'.
142 records inserted into 'books'.
All data inserted successfully!
Database connection closed!
Database loading completed successfully!


In [7]:
# availability_dataframe["id"]=availability_dataframe.index+1
print(availability_dataframe)
print("*"*10)
# category_dataframe["id"]=category_dataframe.index+1
print(category_dataframe)

# print(books_dataframe)

   availability
0          True
**********
         category
0         Mystery
1  Sequential Art
2         Romance


In [ ]:
import data_pipeline.queries_for_analysis as qfa
qfa.execute_checks_and_queries(category_dataframe, availability_dataframe, books_dataframe, clean_data)

Database connected!



01 --------------------------------------------------
Number of records in 'books' table: 142

02 --------------------------------------------------
Number of available books: 0

03 --------------------------------------------------
Number of 5-rated books: 21

04 --------------------------------------------------
Top 5 most expensive books:
                                               title  price_gbp  price_inr
0                 The Perfect Play (Play by Play #1)      59.99   6328.945
1                      Boar Island (Anna Pigeon #19)      59.48   6275.140
2                           Listen to Me (Fusion #1)      58.99   6223.445
3  The No. 1 Ladies' Detective Agency (No. 1 Ladi...      57.70   6087.350
4                                           El Deafo      57.62   6078.910

05 --------------------------------------------------
Average rating by category:
         category  average_rating
0  Sequential Art        2.973333
1         Mystery        2.93750